# `04_Spectral_Clustering.ipynb`

## Topic 1 — Introduction & Intuition

### 1. What is Spectral Clustering?

**Spectral Clustering is a clustering technique that uses the relationships between data points, represented as a graph, to form clusters.**

Unlike K-Means, which mainly looks at the distance from points to **centroids**, Spectral Clustering focuses on:

> **Which data points are strongly connected or similar to each other?**

---

## 2. Why do we need Spectral Clustering?

Consider this dataset:

```text
        ● ● ●
      ●       ●
      ●       ●
        ● ● ●


          ● ● ●
        ●       ●
        ●       ●
          ● ● ●
```

Suppose the data forms two curved/ring-like groups.

K-Means tries to divide the data using distances to centroids.

That can fail for **non-convex shapes**.

Spectral Clustering can instead look at the **connections between nearby points**.

---

## 3. Main Idea

Spectral Clustering works through a graph.

Suppose we have:

```text
A ●────● B
  │    │
  │    │
C ●────● D
```

Each data point becomes a **node**.

Connections between similar/nearby points become **edges**.

So instead of directly thinking:

```text
Points → Centroids → Clusters
```

we think:

```text
Points
  ↓
Graph
  ↓
Connections / Similarities
  ↓
Graph structure
  ↓
Clusters
```

---

## 4. What is the intuition?

Imagine that points belonging to the same cluster are strongly connected:

```text
Cluster 1

A ●────● B
  ╲    ╱
   ╲  ╱
    ● C
```

while two different clusters have weak or no connections:

```text
Cluster 1                 Cluster 2

A ●────● B                 E ●────● F
  ╲    ╱                     ╲    ╱
   ● C                        ● G
```

There is a gap between them.

Spectral Clustering tries to discover these **groups of strongly connected points**.

---

## 5. Why is it called "Spectral"?

The word **spectral** comes from the use of the **eigenvalues and eigenvectors of a matrix derived from the graph**.

Later we'll construct:

* Similarity/Affinity Matrix
* Degree Matrix
* Graph Laplacian

and then perform **eigen-decomposition**.

The important idea for now is:

```text
Data
 ↓
Graph
 ↓
Matrix representation
 ↓
Eigenvectors
 ↓
New representation
 ↓
Clustering
```

We don't need to understand the eigen-decomposition yet. That's a later topic.

---

## 6. Spectral Clustering vs K-Means

### K-Means

Looks mainly at:

> **Distance to centroids**

```text
Points → Centroids → Clusters
```

### Spectral Clustering

Looks mainly at:

> **Relationships between points**

```text
Points → Similarity Graph → Clusters
```

Therefore Spectral Clustering can handle certain cluster shapes that K-Means cannot.

---

## 7. Core Intuition

The most important thing to remember at this stage:

> **Spectral Clustering converts the clustering problem into a graph problem.**

For example:

```text
Original data

● ● ●          ● ● ●
● ● ●          ● ● ●
● ● ●          ● ● ●

       ↓

Graph

●──●          ●──●
│╲ │          │╲ │
●──●          ●──●
│  │          │  │
●──●          ●──●

       ↓

Identify strongly connected groups

       ↓

Cluster 1       Cluster 2
```

So the overall intuition is:

$$\text{Data points}\rightarrow\text{Similarity graph}\rightarrow\text{Graph structure}\rightarrow\text{Clusters}$$

That's the foundation of Spectral Clustering.

# Topic 2 — Graphs, Affinity & Similarity Matrices

Now we convert our dataset into a **graph representation**.

## 1. Represent Data as a Graph

Suppose we have four points:

```text id="x5c1v9"
A ●────● B

C ●────● D
```

We represent:

* **Data point → Node**
* **Similarity/connection → Edge**

So:

```text id="n5k0zq"
A, B, C, D → Nodes

A-B, C-D → Edges
```

The graph tells us **which points are related to each other**.

---

## 2. What Determines an Edge?

We need some way to measure how similar two points are.

For example, suppose two points are close:

```text id="1p4f2e"
A ●────● B
```

They may receive a **high similarity**.

If they are far apart:

```text id="m5h9ve"
A ●                    ● B
```

they may receive a **low similarity**.

So we define:

$$W_{ij}=\text{similarity between points }i\text{ and }j$$

This $W$ is called the **Affinity Matrix** or **Similarity Matrix**.

---

# 3. Affinity Matrix

Suppose we have three points:

```text id="6ty8be"
A, B, C
```

Their similarities might be:

```text
A-B → 0.9
A-C → 0.1
B-C → 0.2
```

Then the affinity matrix could be:

$$
W =
\begin{bmatrix}
0 & 0.9 & 0.1 \\
0.9 & 0 & 0.2 \\
0.1 & 0.2 & 0
\end{bmatrix}
$$

Each entry tells us how strongly two points are connected.

For example:

$$W_{12}=0.9$$

means A and B have high similarity.

And:

$$W_{13}=0.1$$

means A and C have low similarity.

---

# 4. Why Are the Diagonal Values Usually 0?

Look at:

$$W_{11}=0$$

This represents the similarity of A with itself.

In many graph constructions used for spectral clustering, we don't need a self-edge, so the diagonal is set to zero.

Therefore:

$$W_{ii}=0$$

for this common representation.

---

# 5. Symmetry

Usually, similarity is symmetric.

If A is similar to B with value $0.9$, then B is similarly related to A with the same value:

$$W_{ij}=W_{ji}$$

Therefore:

$$
W=\begin{bmatrix}
0&0.9&0.1\\
0.9&0&0.2\\
0.1&0.2&0
\end{bmatrix}
$$

is symmetric.

---

# 6. How Do We Calculate Similarity?

There are several ways.

One common approach is to first calculate **distance**.

For example, Euclidean distance:

$$d(x_i,x_j)=\sqrt{\sum_{k=1}^{n}(x_{ik}-x_{jk})^2}$$

Then we can convert distance into similarity.

One common similarity function is the Gaussian/RBF similarity:

$$W_{ij}=\exp\left(-\frac{|x_i-x_j|^2}{2\sigma^2}\right)$$

Here:

* $|x_i-x_j|$ → distance between the points
* $\sigma$ → controls how quickly similarity decreases with distance

So:

```text
Small distance
      ↓
High similarity

Large distance
      ↓
Low similarity
```

---

# 7. Simple Numerical Example

Suppose:

$$x_1=(1,1)$$

and:

$$x_2=(2,1)$$

Their Euclidean distance is:

$$d(x_1,x_2)=\sqrt{(1-2)^2+(1-1)^2}=1$$

Suppose:

$$\sigma=1$$

Then:

$$W_{12}=\exp\left(-\frac{1^2}{2(1)^2}\right)\approx0.607$$

So the similarity between the two points is approximately:

$$W_{12}\approx0.607$$

---

# 8. Graph Interpretation

Now we can think of the matrix as a graph.

For:

$$W_{12}=0.9$$

we have a strong connection:

```text
A ●━━━━━━━━● B
```

For:

$$W_{13}=0.1$$

we have a weak connection:

```text
A ●────● C
```

Thus the matrix contains all the information about the graph's connections.

---

# 9. Why Is This Important?

This is the foundation of Spectral Clustering.

The process is:

$$\text{Data}\rightarrow\text{Similarity}\rightarrow\text{Affinity Matrix}\rightarrow\text{Graph}$$

Then later:

$$\text{Graph}\rightarrow\text{Graph Laplacian}\rightarrow\text{Eigenvectors}\rightarrow\text{Clustering}$$

So the **Affinity Matrix is the bridge between the original data and the graph-based mathematical formulation**.


# Topic 3 — Graph Laplacian & Eigen-decomposition

Now we reach the **core mathematical part** of Spectral Clustering.

We already have the **Affinity Matrix** $W$.

The next step is to construct the **Graph Laplacian**.

---

## 1. Start with the Affinity Matrix

Suppose we have 3 points:

$$
W=\begin{bmatrix}
0 & 0.8 & 0.2 \\
0.8 & 0 & 0.1 \\
0.2 & 0.1 & 0
\end{bmatrix}
$$

Remember:

* $W_{ij}$ = strength of connection between points $i$ and $j$
* Large value → strong connection
* Small value → weak connection

---

## 2. Calculate the Degree Matrix

For every node, we calculate its total connection strength.

For point 1:

$$D_{11}=0+0.8+0.2=1.0$$

For point 2:

$$D_{22}=0.8+0+0.1=0.9$$

For point 3:

$$D_{33}=0.2+0.1+0=0.3$$

We put these values on the diagonal:

$$
D=\begin{bmatrix}
1.0 & 0 & 0 \\
0 & 0.9 & 0 \\
0 & 0 & 0.3
\end{bmatrix}
$$

All off-diagonal values are zero.

So:

$$D_{ij}=0\quad\text{for }i\neq j$$

---

# 3. What Does the Degree Mean Graphically?

Suppose:

```text
       0.8
  A ━━━━━━━ B
  ╲         ╱
0.2╲       ╱0.1
    ╲     ╱
      C
```

The degree of A is the **total strength of all edges connected to A**:

$$D_{AA}=0.8+0.2=1.0$$

So you can think of:

> **Degree = total connection strength of a node.**

A highly connected node has a large degree.

---

# 4. Graph Laplacian

The basic Graph Laplacian is:

$$L=D-W$$

For our example:

$$
L=
\begin{bmatrix}
1.0 & 0 & 0 \\
0 & 0.9 & 0 \\
0 & 0 & 0.3
\end{bmatrix}
-
\begin{bmatrix}
0 & 0.8 & 0.2 \\
0.8 & 0 & 0.1 \\
0.2 & 0.1 & 0
\end{bmatrix}
$$

Therefore:

$$
L=
\begin{bmatrix}
1.0 & -0.8 & -0.2 \\
-0.8 & 0.9 & -0.1 \\
-0.2 & -0.1 & 0.3
\end{bmatrix}
$$

Notice what happened:

* Diagonal → degree
* Off-diagonal → negative similarity

---

# 5. Why Do We Need the Laplacian?

The affinity matrix tells us:

> **Who is connected to whom?**

The Laplacian gives us a mathematical representation of the **structure of those connections**.

It is especially useful because its eigenvectors reveal information about how the graph can be separated into groups.

That's why the algorithm proceeds:

$$W\rightarrow D\rightarrow L$$

and then:

$$L\rightarrow\text{Eigen-decomposition}$$

---

# 6. Eigen-decomposition

Now we solve:

$$Lv=\lambda v$$

where:

* $L$ = Graph Laplacian
* $v$ = eigenvector
* $\lambda$ = corresponding eigenvalue

We obtain multiple eigenvalue-eigenvector pairs:

$$\lambda_1,v_1$$

$$\lambda_2,v_2$$

$$\lambda_3,v_3$$

and so on.

---

# 7. Which Eigenvectors Do We Use?

Suppose we want:

$$k=2$$

clusters.

We generally take the eigenvectors corresponding to the **smallest $k$ eigenvalues**.

For example, suppose:

$$\lambda_1=0,\quad\lambda_2=0.2,\quad\lambda_3=1.7$$

For $k=2$, we take:

$$v_1,v_2$$

and use them to create a new representation of the points.

---

# 8. Why the Smallest Eigenvalues?

This is the key intuition.

If two groups of nodes have **strong connections inside their groups** but weak connections between groups, the graph has a natural separation.

The smallest eigenvalues/eigenvectors capture these large-scale connectivity structures.

For a graph with perfectly disconnected components, the number of zero eigenvalues of the Laplacian equals the number of connected components.

For example, if:

```text
Cluster 1        Cluster 2

A ─── B          C ─── D
```

with **no edge between the groups**, the graph has two connected components.

Then the Laplacian has two zero eigenvalues:

$$\lambda_1=0,\quad\lambda_2=0$$

This is one of the fundamental reasons spectral clustering works.

---

# 9. What Happens After Eigenvectors?

This is important because **eigenvectors themselves aren't the final clusters**.

Suppose we want $k=2$ clusters.

We take the first two relevant eigenvectors and form a new matrix:

$$U=\begin{bmatrix}v_1&v_2\end{bmatrix}$$

Each original data point now gets a new representation based on its row in $U$.

Then we apply **K-Means to this new representation**.

So the complete mathematical flow is:

$$X\rightarrow W\rightarrow D\rightarrow L\rightarrow U\rightarrow\text{K-Means}\rightarrow\text{Clusters}$$

This is the central idea of Spectral Clustering.

---

### One thing to remember now

Don't worry about manually calculating eigenvectors yet.

At this stage, remember:

$$\boxed{L=D-W}$$

and:

$$\boxed{Lv=\lambda v}$$


# Complete Spectral Clustering Example

We want to see the complete flow:

$$
X\rightarrow W\rightarrow D\rightarrow L\rightarrow\text{Eigenvalues/Eigenvectors}\rightarrow U\rightarrow\text{K-Means}\rightarrow\text{Clusters}
$$

---

## Step 1 — Original Data

Suppose we have **4 data points**:

$$
X=
\begin{bmatrix}
1&1\\
1&2\\
8&8\\
8&9
\end{bmatrix}
$$

The points are:

```text
Point 1 → (1, 1)
Point 2 → (1, 2)
Point 3 → (8, 8)
Point 4 → (8, 9)
```

For this example, we want:

$$k=2$$

clusters.

Visually, we can see two groups, but remember: **in an actual unsupervised problem, we don't give these groups to the algorithm beforehand.**

---

# Step 2 — Construct the Similarity Matrix $W$

For this simple example, we'll use a manually defined graph rather than calculating RBF similarity.

We assume:

* Point 1 and Point 2 have a strong connection.
* Point 3 and Point 4 have a strong connection.
* There are no connections between these two groups.

So the similarity matrix is:

$$
W=
\begin{bmatrix}
0&1&0&0\\
1&0&0&0\\
0&0&0&1\\
0&0&1&0
\end{bmatrix}
$$

This represents the graph:

```text
1 ─── 2        3 ─── 4
```

The `1` means a strong connection.

The `0` means there is no connection.

So:

$$W_{12}=1$$

means Point 1 and Point 2 are strongly connected.

And:

$$W_{13}=0$$

means Point 1 and Point 3 have no connection.

---

# Step 3 — Calculate the Degree Matrix $D$

The degree of a point is the **sum of the connection strengths in its row of $W$**.

For Point 1:

$$D_{11}=0+1+0+0=1$$

For Point 2:

$$D_{22}=1+0+0+0=1$$

For Point 3:

$$D_{33}=0+0+0+1=1$$

For Point 4:

$$D_{44}=0+0+1+0=1$$

Therefore:

$$
D=
\begin{bmatrix}
1&0&0&0\\
0&1&0&0\\
0&0&1&0\\
0&0&0&1
\end{bmatrix}
$$

Notice that the degree values are placed **only on the diagonal**.

---

# Step 4 — Calculate the Graph Laplacian $L$

We already know:

$$L=D-W$$

Substitute $D$ and $W$:

$$
L=
\begin{bmatrix}
1&0&0&0\\
0&1&0&0\\
0&0&1&0\\
0&0&0&1
\end{bmatrix}
-
\begin{bmatrix}
0&1&0&0\\
1&0&0&0\\
0&0&0&1\\
0&0&1&0
\end{bmatrix}
$$

Therefore:

$$
L=
\begin{bmatrix}
1&-1&0&0\\
-1&1&0&0\\
0&0&1&-1\\
0&0&-1&1
\end{bmatrix}
$$

Up to this point, the process is:

$$W\rightarrow D\rightarrow L$$

You already understand this part.

---

# Step 5 — Find the Eigenvalues

Now we work with the Laplacian.

We solve:

$$\det(L-\lambda I)=0$$

For this particular Laplacian, the eigenvalues are:

$$
\lambda_1=0,\quad
\lambda_2=0,\quad
\lambda_3=2,\quad
\lambda_4=2
$$

So, sorted from smallest to largest:

$$0,\quad0,\quad2,\quad2$$

### Why are there two zero eigenvalues?

Look at our graph:

```text
1 ─── 2        3 ─── 4
```

There are **two disconnected components**:

```text
Component 1 → Points 1, 2
Component 2 → Points 3, 4
```

A fundamental property of the Graph Laplacian is:

$$
\text{Number of zero eigenvalues}
=
\text{Number of connected components}
$$

Therefore:

$$
2\text{ zero eigenvalues}
\Rightarrow
2\text{ connected components}
$$

This is already giving us information about the clusters.

---

# Step 6 — Find the Eigenvectors

We now need the eigenvectors corresponding to the two smallest eigenvalues.

For:

$$\lambda_1=0$$

one corresponding eigenvector is:

$$
v_1=
\begin{bmatrix}
1\\
1\\
0\\
0
\end{bmatrix}
$$

For:

$$\lambda_2=0$$

another corresponding eigenvector is:

$$
v_2=
\begin{bmatrix}
0\\
0\\
1\\
1
\end{bmatrix}
$$

Look at what these vectors are telling us.

### Eigenvector $v_1$

$$
v_1=
\begin{bmatrix}
1\\
1\\
0\\
0
\end{bmatrix}
$$

So:

```text
Point 1 → 1
Point 2 → 1
Point 3 → 0
Point 4 → 0
```

### Eigenvector $v_2$

$$
v_2=
\begin{bmatrix}
0\\
0\\
1\\
1
\end{bmatrix}
$$

So:

```text
Point 1 → 0
Point 2 → 0
Point 3 → 1
Point 4 → 1
```

Notice that the eigenvectors have already separated the two groups.

But we still need to create the new representation.

---

# Step 7 — Construct the Eigenvector Matrix $U$

We take the selected eigenvectors and place them **as columns**.

$$
U=
\begin{bmatrix}
|&|\\
v_1&v_2\\
|&|
\end{bmatrix}
$$

Therefore:

$$
U=
\begin{bmatrix}
1&0\\
1&0\\
0&1\\
0&1
\end{bmatrix}
$$

Now comes an **extremely important point**:

> **Each row of $U$ corresponds to one original data point.**

So:

$$\text{Point 1}\rightarrow[1,0]$$

$$\text{Point 2}\rightarrow[1,0]$$

$$\text{Point 3}\rightarrow[0,1]$$

$$\text{Point 4}\rightarrow[0,1]$$

Therefore, the new representation is:

$$
X'=
\begin{bmatrix}
1&0\\
1&0\\
0&1\\
0&1
\end{bmatrix}
$$

---

# Step 8 — Why Is This New Representation Useful?

This is the **main purpose of the eigenvectors**.

Originally, our points were:

```text
Point 1 → (1,1)
Point 2 → (1,2)

Point 3 → (8,8)
Point 4 → (8,9)
```

After spectral transformation, they become:

```text
Point 1 → (1,0)
Point 2 → (1,0)

Point 3 → (0,1)
Point 4 → (0,1)
```

So in the new spectral space:

```text
Point 1 ── Point 2

Point 3 ── Point 4
```

The points belonging to the same graph component have **identical representations**.

This makes the clustering extremely easy.

---

# Step 9 — Apply K-Means

Now we finally use K-Means.

But there is an important detail:

> **We apply K-Means to $U$, not to the original $X$.**

We have:

$$
U=
\begin{bmatrix}
1&0\\
1&0\\
0&1\\
0&1
\end{bmatrix}
$$

and:

$$k=2$$

K-Means sees these four points:

```text
[1, 0]
[1, 0]
[0, 1]
[0, 1]
```

It groups the similar representations:

```text
Cluster 1:
[1,0]
[1,0]

Cluster 2:
[0,1]
[0,1]
```

Therefore:

```text
Point 1 → Cluster 1
Point 2 → Cluster 1
Point 3 → Cluster 2
Point 4 → Cluster 2
```

The actual labels could also be reversed:

```text
Point 1 → Cluster 2
Point 2 → Cluster 2
Point 3 → Cluster 1
Point 4 → Cluster 1
```

That would represent **exactly the same clustering**. Cluster numbers themselves have no inherent meaning.

---

# Complete Flow

### 1. Original data

$$
X=
\begin{bmatrix}
1&1\\
1&2\\
8&8\\
8&9
\end{bmatrix}
$$

↓

### 2. Similarity Matrix

$$
W=
\begin{bmatrix}
0&1&0&0\\
1&0&0&0\\
0&0&0&1\\
0&0&1&0
\end{bmatrix}
$$

↓

### 3. Degree Matrix

$$
D=
\begin{bmatrix}
1&0&0&0\\
0&1&0&0\\
0&0&1&0\\
0&0&0&1
\end{bmatrix}
$$

↓

### 4. Graph Laplacian

$$
L=D-W=
\begin{bmatrix}
1&-1&0&0\\
-1&1&0&0\\
0&0&1&-1\\
0&0&-1&1
\end{bmatrix}
$$

↓

### 5. Eigenvalues

$$0,\quad0,\quad2,\quad2$$

↓

### 6. Select eigenvectors corresponding to the two smallest eigenvalues

$$
v_1=
\begin{bmatrix}
1\\
1\\
0\\
0
\end{bmatrix},
\quad
v_2=
\begin{bmatrix}
0\\
0\\
1\\
1
\end{bmatrix}
$$

↓

### 7. Construct $U$

$$
U=
\begin{bmatrix}
1&0\\
1&0\\
0&1\\
0&1
\end{bmatrix}
$$

↓

### 8. Use $U$ as the new dataset

```text
Point 1 → (1,0)
Point 2 → (1,0)
Point 3 → (0,1)
Point 4 → (0,1)
```

↓

### 9. K-Means

$$U\rightarrow\text{K-Means with }k=2$$

↓

### 10. Final clusters

```text
Cluster 1 → Points 1, 2
Cluster 2 → Points 3, 4
```

---

## The one thing to remember

The whole purpose of the spectral part is:

$$
X
\rightarrow
\text{Graph}
\rightarrow
\text{Eigenvector representation}
\rightarrow
\text{Easy clustering}
$$

The **eigenvectors don't directly give us final cluster labels**. They create a **new representation $U$** in which the graph-based structure is easier for K-Means to separate.

# Topic 4 — Spectral Clustering Algorithm

Now we put everything we've learned together into the **actual algorithm**.

We'll go **one step at a time**.

---

## Step 1 — Start with the Data

We have our dataset:

$$
X=
\begin{bmatrix}
x_1\
x_2\
\vdots\
x_n
\end{bmatrix}
$$

and we decide how many clusters we want:

$$
k=\text{number of clusters}
$$

For example:

$$
k=2
$$

At this point, we **don't know which point belongs to which cluster**.

---

## Step 2 — Build the Similarity / Affinity Matrix

We calculate how similar every pair of points is.

$$
W_{ij}=\text{similarity}(x_i,x_j)
$$

For example, using RBF similarity:

$$
W_{ij}=
\exp\left(-\frac{\lVert x_i-x_j\rVert^2}{2\sigma^2}\right)
$$

This gives us the matrix:

$$
W=
\begin{bmatrix}
W_{11}&W_{12}&\cdots&W_{1n}\
W_{21}&W_{22}&\cdots&W_{2n}\
\vdots&\vdots&\ddots&\vdots\
W_{n1}&W_{n2}&\cdots&W_{nn}
\end{bmatrix}
$$

So now we have a **graph representation** of our data.

---

## Step 3 — Build the Degree Matrix

For each point, calculate the total strength of its connections:

$$
D_{ii}=\sum_j W_{ij}
$$

Then put these values on the diagonal:

$$
D=
\begin{bmatrix}
D_{11}&0&\cdots&0\
0&D_{22}&\cdots&0\
\vdots&\vdots&\ddots&\vdots\
0&0&\cdots&D_{nn}
\end{bmatrix}
$$

---

## Step 4 — Construct the Graph Laplacian

For the basic unnormalized version:

$$
L=D-W
$$

This is the matrix that captures the **structure of the graph**.

We've already studied this part in detail.

---

## Step 5 — Find Eigenvalues and Eigenvectors

Now solve:

$$
Lv=\lambda v
$$

This gives us:

* eigenvalues $\lambda$
* corresponding eigenvectors $v$

Sort the eigenvalues from smallest to largest.

For example:

$$
\lambda_1\leq\lambda_2\leq\lambda_3\leq\cdots\leq\lambda_n
$$

---

## Step 6 — Select the Smallest $k$ Eigenvectors

If we want:

$$
k=2
$$

we select the eigenvectors corresponding to:

$$
\lambda_1,\lambda_2
$$

Call them:

$$
v_1,v_2
$$

Then put them together as columns:

$$
U=
\begin{bmatrix}
v_1&v_2
\end{bmatrix}
$$

For general $k$:

$$
U=
\begin{bmatrix}
v_1&v_2&\cdots&v_k
\end{bmatrix}
$$

---

## Step 7 — Create the New Representation

This is the **important transformation**.

The original data was:

$$
X
$$

Now we represent each point using its corresponding row of $U$.

If:

$$
U=
\begin{bmatrix}
0.8&0.1\
0.7&0.2\
0.1&0.9\
0.2&0.8
\end{bmatrix}
$$

then:

```text
Point 1 → (0.8, 0.1)
Point 2 → (0.7, 0.2)
Point 3 → (0.1, 0.9)
Point 4 → (0.2, 0.8)
```

Notice what happened.

The original features are **no longer being used directly for the final clustering**.

We now have a new space based on the **graph structure**.

---

## Step 8 — Apply K-Means

Now we run K-Means on $U$.

**Not on the original $X$.**

$$
U\rightarrow\text{K-Means}\rightarrow\text{Cluster Labels}
$$

If:

$$
k=2
$$

K-Means finds two groups in this new eigenvector space.

For example:

```text
Point 1 → Cluster 0
Point 2 → Cluster 0
Point 3 → Cluster 1
Point 4 → Cluster 1
```

---

# Complete Algorithm

So the entire Spectral Clustering algorithm is:

$$
X\rightarrow W\rightarrow D\rightarrow L\rightarrow\text{Eigenvalues/Eigenvectors}\rightarrow U\rightarrow\text{K-Means}\rightarrow\text{Clusters}
$$

Or in simple words:

```text
Original Data
     ↓
Calculate Similarities
     ↓
Build Graph
     ↓
Affinity Matrix W
     ↓
Degree Matrix D
     ↓
Laplacian L = D - W
     ↓
Find Eigenvalues + Eigenvectors
     ↓
Take k smallest eigenvalue eigenvectors
     ↓
Create U
     ↓
Use rows of U as new data
     ↓
K-Means
     ↓
Final Clusters
```

### The key idea

**Spectral Clustering does not directly cluster the original points.**

It first uses the **relationships between points** to create a new representation $U$.

Then K-Means clusters that new representation.

$$
\boxed{\text{Graph structure}\rightarrow\text{new representation}\rightarrow\text{clustering}}
$$


# Topic 5 — Important Parameters

For Spectral Clustering in Scikit-learn, there are several important parameters. But let's first understand **what each parameter controls conceptually**.

The main constructor is:

```python
from sklearn.cluster import SpectralClustering

model = SpectralClustering(
    n_clusters=2,
    affinity="rbf",
    gamma=1.0,
    assign_labels="kmeans"
)
```

We'll go **one parameter at a time**.

---

## 1. `n_clusters`

This tells Spectral Clustering **how many final clusters you want**.

```python
n_clusters=2
```

For example:

```text
n_clusters = 2
→ find 2 clusters

n_clusters = 3
→ find 3 clusters
```

Mathematically, this determines how many eigenvectors are selected:

$$k=n_clusters$$

So if:

$$k=2$$

we use the eigenvectors corresponding to the two smallest eigenvalues.

### Important

Unlike DBSCAN, Spectral Clustering **requires you to specify the number of clusters**.

```text
DBSCAN
→ does not require number of clusters

Spectral Clustering
→ requires number of clusters
```

---

## 2. `affinity`

This tells the algorithm **how to construct the similarity/affinity graph**.

Common options include:

```python
affinity="rbf"
```

or:

```python
affinity="nearest_neighbors"
```

### `rbf`

Uses RBF/Gaussian similarity:

$$W_{ij}=\exp\left(-\gamma|x_i-x_j|^2\right)$$

Here $\gamma$ controls how quickly similarity decreases with distance.

### `nearest_neighbors`

Instead of connecting every point to every other point, it builds the graph using nearby points.

For example:

```python
affinity="nearest_neighbors"
```

uses:

```python
n_neighbors=10
```

to determine how many neighbors are considered.

---

## 3. `gamma`

`gamma` is important when:

```python
affinity="rbf"
```

It controls how quickly similarity decreases as distance increases.

The formula used by Scikit-learn is:

$$W_{ij}=\exp\left(-\gamma|x_i-x_j|^2\right)$$

### Large `gamma`

Similarity decreases quickly.

```text
Large gamma
     ↓
Only nearby points strongly connected
     ↓
More local graph
```

### Small `gamma`

Similarity decreases slowly.

```text
Small gamma
     ↓
Farther points can still have similarity
     ↓
More globally connected graph
```

This is related to the $\sigma$ we discussed earlier.

The relationship is:

$$\gamma=\frac{1}{2\sigma^2}$$

So:

```text
Large gamma ↔ Small sigma
Small gamma ↔ Large sigma
```

---

## 4. `n_neighbors`

This is important when:

```python
affinity="nearest_neighbors"
```

It determines how many neighboring points are used to construct the graph.

For example:

```python
n_neighbors=5
```

means the graph is constructed using the nearest 5 neighbors.

### Small value

```text
Small n_neighbors
→ local connections
→ sparse graph
```

### Large value

```text
Large n_neighbors
→ more connections
→ denser graph
```

---

## 5. `assign_labels`

After obtaining the spectral representation, we still need to assign points to clusters.

This parameter determines **how the final labels are assigned**.

Common choices:

```python
assign_labels="kmeans"
```

```python
assign_labels="discretize"
```

```python
assign_labels="cluster_qr"
```

The default is generally:

```python
assign_labels="kmeans"
```

With K-Means:

$$U\rightarrow\text{K-Means}\rightarrow\text{Cluster Labels}$$

This is the step we already discussed.

---

### For your learning

The parameters you should understand most deeply are:

```text
n_clusters
    ↓
How many clusters?

affinity
    ↓
How do we construct the graph?

gamma
    ↓
How quickly RBF similarity decreases?

n_neighbors
    ↓
How many nearby points are connected?

assign_labels
    ↓
How do we convert spectral representation into final labels?
```


### $\gamma$ controls the **range of influence of each point**

It decides **how far a point can have a meaningful similarity/connection with other points**.

The RBF similarity is:

$$W_{ij}=\exp\left(-\gamma|x_i-x_j|^2\right)$$

Think of each point as having an **influence radius**.

### Small $\gamma$

The influence is **wide**.

```text
Point A
   ↓
Nearby points       → strong similarity
Moderately far      → still noticeable similarity
Far points          → may still have some similarity
```

So the graph becomes **more connected**.

### Large $\gamma$

The influence is **narrow**.

```text
Point A
   ↓
Very nearby points  → strong similarity
Moderately far      → weak similarity
Far points          → almost zero similarity
```

So the graph becomes **more local/sparser**.

### What actually changes?

Suppose the points don't change at all:

```text
A -------- B -------- C
```

Changing $\gamma$ changes the **edge weights**:

$$\gamma\uparrow\Rightarrow W_{ij}\downarrow\text{ faster as distance increases}$$

$$\gamma\downarrow\Rightarrow W_{ij}\downarrow\text{ more slowly as distance increases}$$

Therefore:

**$\gamma$ does NOT change the data points or their distances.**

It changes **how strongly the algorithm considers two points connected based on their distance**.

And because $W$ changes, everything after it can change:

$$\gamma\rightarrow W\rightarrow D\rightarrow L\rightarrow\text{Eigenvectors}\rightarrow\text{Clusters}$$

So the simplest definition is:

> **Gamma controls the effective neighborhood size in the RBF similarity graph.**

* **Small $\gamma$ → larger effective neighborhood**
* **Large $\gamma$ → smaller effective neighborhood**


## Topic 6 — Scikit-Learn Implementation

Now we'll use `SpectralClustering` in Scikit-learn.

We'll follow your dedicated structure.

### 1. Introduction

Scikit-learn provides Spectral Clustering through:

```python
from sklearn.cluster import SpectralClustering
```

The basic workflow is:

```text
Data
 ↓
SpectralClustering()
 ↓
fit_predict()
 ↓
Cluster labels
```

Unlike our manual implementation, Scikit-learn handles internally:

* affinity graph construction
* graph Laplacian
* eigenvalue/eigenvector calculation
* spectral representation
* final label assignment

So we don't manually calculate $W$, $D$, $L$, eigenvectors, and K-Means when using the estimator.

---

### 2. Import & Constructor

Import:

```python
from sklearn.cluster import SpectralClustering
```

Create the model:

```python
model = SpectralClustering(
    n_clusters=2,
    affinity="rbf",
    gamma=1.0,
    assign_labels="kmeans",
    random_state=42
)
```

At this point, we have **only created the model**.

Clustering has not happened yet.

The clustering happens when we call:

```python
labels = model.fit_predict(X)
```

For example, the output might be:

```text
[0 0 1 1]
```

Meaning:

```text
Point 1 → Cluster 0
Point 2 → Cluster 0
Point 3 → Cluster 1
Point 4 → Cluster 1
```

The actual cluster numbers can be reversed; `0` and `1` themselves have no inherent meaning.


## 3. Parameters — `SpectralClustering`

The important parameters are:

```python
model = SpectralClustering(
    n_clusters=2,
    affinity="rbf",
    gamma=1.0,
    n_neighbors=10,
    assign_labels="kmeans",
    random_state=42
)
```

### `n_clusters`

Number of clusters to create.

```python
n_clusters=2
```

If:

```text
n_clusters = 2 → 2 clusters
n_clusters = 3 → 3 clusters
```

This is also the $k$ used when selecting the spectral representation.

---

### `affinity`

Specifies **how the similarity graph is constructed**.

Common choices:

```python
affinity="rbf"
```

or:

```python
affinity="nearest_neighbors"
```

#### `rbf`

Uses RBF similarity:

$$W_{ij}=\exp\left(-\gamma|x_i-x_j|^2\right)$$

Here `gamma` controls how quickly the similarity decreases with distance.

#### `nearest_neighbors`

Builds the graph using the nearest neighbors of each point.

In this case, `n_neighbors` becomes important.

---

### `gamma`

Used when:

```python
affinity="rbf"
```

It controls the **effective neighborhood size** of the RBF graph.

```text
Small gamma
→ similarity decreases slowly
→ wider connections

Large gamma
→ similarity decreases quickly
→ more local connections
```

So:

$$\gamma\uparrow\Rightarrow\text{effective neighborhood}\downarrow$$

and:

$$\gamma\downarrow\Rightarrow\text{effective neighborhood}\uparrow$$

---

### `n_neighbors`

Mainly used when:

```python
affinity="nearest_neighbors"
```

It determines how many nearest neighbors are considered when constructing the graph.

```python
n_neighbors=10
```

means the graph is constructed using the nearest 10 neighbors.

```text
Small n_neighbors
→ fewer connections
→ more local graph

Large n_neighbors
→ more connections
→ denser graph
```

---

### `assign_labels`

Determines how the spectral embedding is converted into final cluster labels.

Common options include:

```python
assign_labels="kmeans"
```

```python
assign_labels="discretize"
```

```python
assign_labels="cluster_qr"
```

With the default K-Means approach:

$$U\rightarrow\text{K-Means}\rightarrow\text{Cluster Labels}$$

---

### `random_state`

Controls randomness so that you can get reproducible results.

```python
random_state=42
```

It is particularly useful when `assign_labels="kmeans"` because K-Means has random initialization.

---

### What you should remember

```text
n_clusters
→ How many clusters?

affinity
→ How is the graph/similarity constructed?

gamma
→ How local is the RBF graph?

n_neighbors
→ How many neighbors are connected?

assign_labels
→ How are final labels assigned?

random_state
→ Make random results reproducible
```

These are the parameters we'll actually use in the implementation.


## 4. Methods & Attributes — `SpectralClustering`

Now we look at **what we can call on the model and what information we can access after fitting it**.

---

### Methods

### 1. `fit(X)`

Fits the Spectral Clustering model to the data.

```python
model.fit(X)
```

It performs the clustering process internally.

After this, the model has been fitted.

---

### 2. `fit_predict(X)`

This is the method you'll use most often.

```python
labels = model.fit_predict(X)
```

It:

1. Fits the model.
2. Calculates the clustering.
3. Returns the cluster label for every point.

For example:

```text id="7j1m3f"
labels
→ [0, 0, 1, 1]
```

---

### 3. `get_params()`

Returns the parameters currently used by the model.

```python id="y9v1n5"
model.get_params()
```

For example, it can show:

```text id="y6z9la"
n_clusters
affinity
gamma
n_neighbors
assign_labels
random_state
...
```

This is useful for checking the model configuration.

---

### 4. `set_params()`

Used to change parameters after creating the model.

For example:

```python id="k4n5qf"
model.set_params(
    n_clusters=3
)
```

Now the model is configured to use 3 clusters.

You can also change multiple parameters:

```python id="m2g4qs"
model.set_params(
    n_clusters=3,
    gamma=0.5
)
```

---

# Attributes

### `labels_`

After fitting, this contains the cluster assigned to each sample.

```python id="z8xj4m"
model.labels_
```

Example:

```text id="b7m0z1"
[0, 0, 1, 1]
```

The order corresponds to the order of your input data.

If:

```text id="j5p2x8"
X[0] → Point 1
X[1] → Point 2
X[2] → Point 3
X[3] → Point 4
```

then:

```text id="h0c5ks"
labels_[0] → cluster of Point 1
labels_[1] → cluster of Point 2
labels_[2] → cluster of Point 3
labels_[3] → cluster of Point 4
```

---

## Important difference

`fit()`:

```python id="0q5j6p"
model.fit(X)
```

does the fitting but **doesn't directly give you the labels**.

`fit_predict()`:

```python id="0a9p4k"
labels = model.fit_predict(X)
```

fits the model **and returns the labels**.

So for normal clustering work, you'll usually use:

```python id="3w7m5c"
labels = model.fit_predict(X)
```

---

### What to remember

```text id="u4j6p1"
Methods
├── fit()
├── fit_predict()
├── get_params()
└── set_params()

Important Attribute
└── labels_
```


# 6. Complete Code — Spectral Clustering

Here is the complete Scikit-learn implementation combining the workflow we just covered.

```python
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import SpectralClustering


# 1. Data
X = np.array([
    [1, 1],
    [1, 2],
    [8, 8],
    [8, 9]
])


# 2. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# 3. Create Spectral Clustering Model
model = SpectralClustering(
    n_clusters=2,
    affinity="rbf",
    gamma=1.0,
    assign_labels="kmeans",
    random_state=42
)


# 4. Fit + Predict
labels = model.fit_predict(X_scaled)


# 5. Display Cluster Labels
print("Cluster Labels:", labels)


# 6. Visualization
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=labels
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Spectral Clustering")
plt.show()
```

### What each important line does

```python
model = SpectralClustering(
    n_clusters=2,
    affinity="rbf",
    gamma=1.0,
    assign_labels="kmeans",
    random_state=42
)
```

Creates the Spectral Clustering model.

```python
labels = model.fit_predict(X_scaled)
```

This is the **main line**.

It performs the clustering and returns the labels.

```python
print(labels)
```

Shows which cluster each point belongs to.

```python
c=labels
```

uses those labels to visually distinguish the clusters.

### The important thing to remember

You don't need to manually write:

```text
W
D
L
Eigenvalues
Eigenvectors
U
K-Means
```

when using `SpectralClustering`.

Scikit-learn performs those steps internally.

Your code is essentially:

$$
\text{Data}
\rightarrow
\text{SpectralClustering}
\rightarrow
\text{fit\_predict}
\rightarrow
\text{Labels}
$$

## 7. Common Errors — Spectral Clustering

### 1. Choosing `n_clusters` incorrectly

`n_clusters` must be appropriate for the problem.

```python
model = SpectralClustering(
    n_clusters=2
)
```

If you choose an unsuitable number, the algorithm will still produce that many clusters, but the grouping may not be meaningful.

---

### 2. Using `gamma` without understanding its effect

When:

```python
affinity="rbf"
```

`gamma` controls how quickly similarity decreases with distance.

$$\gamma\uparrow\rightarrow\text{more local connections}$$

$$\gamma\downarrow\rightarrow\text{wider connections}$$

A poor value can produce an unsuitable affinity graph and therefore poor clusters.

---

### 3. Using `n_neighbors` with the wrong affinity

`n_neighbors` is relevant when:

```python
affinity="nearest_neighbors"
```

For example:

```python
model = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    n_neighbors=10
)
```

If you're using:

```python
affinity="rbf"
```

then `gamma` is the important graph parameter instead.

---

### 4. Forgetting feature scaling

Spectral Clustering relies on distances/similarities.

Suppose one feature ranges from:

```text
0 → 1
```

while another ranges from:

```text
0 → 100000
```

The larger-scale feature can dominate the distance calculation.

Use scaling when appropriate:

```python
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
```

---

### 5. Using too few neighbors

With:

```python
affinity="nearest_neighbors"
```

if `n_neighbors` is too small, the graph can become too sparse.

```text
Too few neighbors
        ↓
Too few connections
        ↓
Graph may become disconnected
        ↓
Poor clustering
```

---

### 6. Using too many neighbors

If `n_neighbors` is excessively large:

```text
Too many neighbors
        ↓
Many connections
        ↓
Different groups may become strongly connected
        ↓
Cluster separation can weaken
```

---

### 7. Forgetting `random_state`

When using:

```python
assign_labels="kmeans"
```

K-Means involves randomness.

Without a fixed seed, repeated runs may produce different results.

Use:

```python
random_state=42
```

when you want reproducible results.

---

### 8. Expecting `labels_` before fitting

This is incorrect:

```python
model = SpectralClustering(n_clusters=2)

print(model.labels_)
```

The model hasn't been fitted yet.

Correct:

```python
model.fit(X)

print(model.labels_)
```

Or simply:

```python
labels = model.fit_predict(X)
```

---

### 9. Too many data points

Spectral Clustering can become computationally expensive for very large datasets because it involves operations on the similarity graph and eigenvectors.

So it is generally more suitable for **small-to-medium-sized datasets** than extremely large datasets.

---

### Most important errors to remember

```text
Wrong scaling
     ↓
Wrong distances

Wrong gamma
     ↓
Wrong RBF graph

Wrong n_neighbors
     ↓
Wrong nearest-neighbor graph

Wrong n_clusters
     ↓
Wrong number of groups

No random_state
     ↓
Possibly different results between runs
```

The next section is **8. Important Notes / Key Takeaways**, which will finish the Scikit-learn section.
